# Package Design and Julia Compatibility

**Abstract.** `tightbinding_py` is a Python port of the Julia package
`TightBinding.jl`. It constructs
real-space tight-binding models on Bravais lattices, builds uniform
(reciprocal / flux) grids, and computes band structures and single-particle /
many-body Chern numbers.

This notebook describes the implementation in `src/tightbinding_py`.
Read the [README](../README.md) for installation and the
[lattice](lattice.ipynb), [model](tightbinding_model.ipynb),
[band-topology](band_topology.ipynb), and [many-body](many_body_chern.ipynb)
notebooks for worked tutorials. Fenced snippets below describe interfaces and
conventions; the final code cell is a runnable consistency check.  A dedicated section at the end lists every
deliberate difference from the Julia source together with the reason for it.

## 1. Package layout

```
src/tightbinding_py/
    __init__.py        # public API re-exports
    lattice.py         # Real_Space_Lattice, LatticeGraph, initialize_real_space_lattice, plot_real_space_lattice
    tb_model.py        # Real_Space_TightBinding_Model, add_hopping_term, add_hoppings_by_graph_distance, ...
    uniform_grids.py   # Uniform_Grids, initialize_uniform_grids, initialize_uniform_grids_from_lattice
    utils.py           # dual basis, build_Hk_crys, Chern numbers, bilinear terms, real-space Hamiltonian
    band_plot.py       # plot_bands, plot_band_contour, find_1st_BZ_k_cart_list
```

Every Julia source file has a one-to-one Python counterpart:

| Julia file | Python file |
|---|---|
| `src/real_space_lattice.jl` | `src/tightbinding_py/lattice.py` |
| `src/real_space_tb_model.jl` | `src/tightbinding_py/tb_model.py` |
| `src/uniform_grids.jl` | `src/tightbinding_py/uniform_grids.py` |
| `src/utils.jl` | `src/tightbinding_py/utils.py` |
| `src/band_plot.jl` | `src/tightbinding_py/band_plot.py` |

## 2. Coordinates

Bravais vectors are stored as **row vectors** (a list of lists):

```python
brav_vec_list = [[a1x, a1y], [a2x, a2y]]      # dim rows of length dim
brav_rows = np.asarray(brav_vec_list, dtype=np.float64)
```

Crystal coordinates are converted to Cartesian coordinates by right-multiplying
the row matrix:

```python
cart = crys @ brav_rows        # crys is a 1-d array of length dim
```

The real-space cell volume is `cell_volume = abs(det(brav_rows))`.

Reciprocal (dual) vectors satisfy `b_i · a_j = 2π δ_ij` and are computed by the
2π-inverse-transpose rule

```python
dual_basis_vec_mat(basis_vec_mat) = 2π * inv(basis_vec_mat).T     # columns
```

so that `dual_basis_vec_mat.T @ basis_vec_mat = 2π I` (both matrices stored **in
columns**; a list input is `hcat`-ed into columns first).  The Bloch phase
identity then reads `k_cart · r_cart = 2π * (k_crys · r_crys)`.

## 3. Real-space lattices (`lattice.py`)

### 3.1 `Real_Space_Lattice`

```python
@dataclass
class Real_Space_Lattice:
    lattice_name: str
    dim: int
    sample_size: list[int]                      # unit cells per direction
    cell_int_list: list[tuple[int, ...]]        # crystal-coordinate cell indices
    n_cell: int
    brav_vec_list: list[list[float]]
    cell_volume: float
    n_sub: int
    sub_crys_list: list[list[float]]            # sublattice positions (crystal coords)
    sub_name_list: list[str]                    # default "A1", "A2", ...
    pbc_indicator: list[bool]
    twisted_phases_over_2π: list[float]
    n_site: int
    site_list: list[Site]                       # Site = ((cell_int,), i_sub)  1-based sub
    site_crys_list: list[np.ndarray]
    site_cart_list: list[np.ndarray]
    site_to_index_map: dict[Site, int]          # 1-BASED site indices
    graph: LatticeGraph | None
```

A site is `(cell_int, i_sub)` where `cell_int` is a **tuple of integers**
(hashable — used directly as a dict key) and `i_sub` is the **1-based**
sublattice index.  All site indices (`site_to_index_map`, graph vertex ids,
`generate_bilinear_terms` output) are **1-based**, exactly as in the Julia
package.

### 3.2 Cell and site enumeration — first axis fastest

```python
cell_int_list = [
    (i0, i1, ...) 
    for iN in range(sample_size[N-1])       # last axis outermost
    for ... 
    for i0 in range(sample_size[0])         # FIRST axis fastest (innermost)
]
site_list = [(cell, i_sub) for cell in cell_int_list for i_sub in range(1, n_sub + 1)]
```

This reproduces Julia's `Iterators.product([0:(Ni-1) for Ni in sample_size]...)`,
so the *first* Bravais axis advances fastest.  For honeycomb `[3, 4]` the head of
`site_list` is

```python
[((0, 0), 1), ((0, 0), 2), ((1, 0), 1), ((1, 0), 2), ((2, 0), 1), ((2, 0), 2)]
```

### 3.3 Construction

```python
initialize_real_space_lattice(
    *, brav_vec_list=[[1.0,0.0],[0.0,1.0]], sample_size=[2,2],
    sub_crys_list=[[0.0,0.0]], lattice_name="", pbc_indicator=[True,True],
    twisted_phases_over_2π=None, allowed_bonds=None,
)
```

- A `lattice_name` that exactly matches `"square"`, `"honeycomb"`, `"kagome"`,
  `"Lieb"`, or `"dice"` **overrides** `brav_vec_list` / `sub_crys_list` (and for
  `"dice"` also `allowed_bonds`) with the preset values — mirroring the Julia
  `@match` block.
- `twisted_phases_over_2π` defaults to zeros and must have length `dim`.
  Non-zero phases are only allowed where `pbc_indicator[d]` is `true`
  (enforced by raising `ValueError`).

The dice preset ships `allowed_bonds = [(1, 2), (2, 3)]` (bonds connecting sublattice 2 to sublattices 1 and 3).

### 3.4 Nearest-neighbour graph

`lattice.graph` is a :class:`LatticeGraph` wrapping an `igraph.Graph` (see the
difference notes in §9).  It is built by the Euclidean-distance algorithm: first
find the minimal inter-site distance (with the minimum-image convention), then
connect every pair whose distance falls within that value (times `1 + 1e-10`).

The minimum-image convention is the `_wrap_Δ_crys` helper:

```python
def _wrap_Δ_crys(Δ_crys, *, sample_size, pbc_indicator):
    for d in range(dim):
        if pbc_indicator[d]:
            Δ_crys[d] -= round(Δ_crys[d] / sample_size[d]) * sample_size[d]
```

i.e. the crystal displacement is wrapped into `[-L/2, L/2]` for every periodic
direction.  This is what turns a finite sample into a torus and produces the
"wrapped" (ghost) bonds drawn by the plotting routines.

`LatticeGraph` exposes the small `Graphs.SimpleGraph` API subset the package
needs, all **1-based**:

```python
g.neighbors(i)            # set[int]
g.gdistances(i)           # dict[int, int]  (BFS distance from i)
g.edges()                 # list[(int, int)] with i < j
g.igraph_graph            # the underlying igraph.Graph (0-based)
```

Numeric Bravais vectors and sublattice positions are required; conversion
to floats occurs before graph construction. If the graph builder itself raises
`TypeError` or `ValueError`, `graph` falls back to `None` (see §9g).

## 4. Tight-binding models (`tb_model.py`)

### 4.1 `Real_Space_TightBinding_Model`

```python
@dataclass
class Real_Space_TightBinding_Model:
    lattice: Real_Space_Lattice
    model_name: str
    input_hopping_map: dict[SitePair, complex]   # translation-invariant templates
    full_hopping_map: dict[SitePair, complex]    # translation-expanded, PBC-wrapped
    H_hop: Any = None                            # parity placeholder (unused)
```

- `input_hopping_map` holds the **minimal** templates the user declared
  (Hermiticity already enforced).
- `full_hopping_map` holds every template **expanded by translation symmetry**
  over all cells (with PBC wrapping).  It is rebuilt after each
  `add_hopping_term`.

A hopping term is the Python analogue of Julia's `Pair{Tuple{Site,Site},T}`:

```python
term = (((cell_from, sub_from), (cell_to, sub_to)), amplitude)   # subs 1-BASED
```

### 4.2 Adding hoppings

```python
add_hopping_term(tb, term, *, is_hermitian=True)
```

writes the term into `input_hopping_map` (overwriting, with a printed warning,
if the key exists) and, when `is_hermitian=True`, also writes the reverse term
with the conjugated amplitude.  It then re-expands `full_hopping_map`.

```python
add_hopping_term_to_full_hopping_map(tb, term, *, is_hermitian=True)
```

adds a single term to `full_hopping_map` **without** translation expansion
(existing keys are summed, not overwritten).

```python
add_hoppings_by_graph_distance(tb, graph_distance, amplitude, *, is_hermitian=True)
```

adds hoppings between **all** site pairs at a given graph distance on the
lattice graph, via `add_hopping_term_to_full_hopping_map`.  `amplitude` may be a
uniform complex number or a callable `amplitude(i_site, j_site) -> complex`
receiving two 1-based linear indices (used for the direction-dependent Haldane
NNN amplitude). `graph_distance = 0` writes on-site terms.

Use template-based or graph-based construction consistently: adding a template
rebuilds `full_hopping_map`, and `build_Hk_crys` uses only templates when
`input_hopping_map` is non-empty.

### 4.3 The Haldane model (canonical template snippet)

This is the exact template set used throughout the tests and examples
(`t1 = -1`, `t2 = -0.24`, `φ = π/2`, `M = 0.7`):

```python
t1, t2, phi, M = -1.0, -0.24, np.pi / 2, 0.7

# staggered sublattice potential (mass); on-site => is_hermitian=False
add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 1)),  M), is_hermitian=False)
add_hopping_term(tb, ((((0, 0), 2), ((0, 0), 2)), -M), is_hermitian=False)

# nearest-neighbour hopping t1 (A1 -> A2, three directed templates)
add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 2)), t1))
add_hopping_term(tb, ((((0, 0), 1), ((0, -1), 2)), t1))
add_hopping_term(tb, ((((0, 0), 1), ((-1, 0), 2)), t1))

# complex next-nearest-neighbour hopping (chirality-dependent phase)
for src in (1, 2):
    sgn = 1 if src == 1 else -1
    add_hopping_term(tb, ((((0, 0), src), ((1, 0), src)), t2 * np.exp(sgn * 1j * phi)))
    add_hopping_term(tb, ((((0, 0), src), ((0, 1), src)), t2 * np.exp(-sgn * 1j * phi)))
    add_hopping_term(tb, ((((0, 0), src), ((-1, 1), src)), t2 * np.exp(sgn * 1j * phi)))
```

With this chirality convention (sublattice `A1` gets `sgn = +1`, `A2` gets
`-1`) the lower band carries Chern number `-1` and the upper band `+1`.

An alternative **graph-based** construction starts with an empty model and
adds all terms through the graph:

```python
def haldane_amp(i, j):
    return haldane_nnn_hopping_amplitude(i, j, t2=-0.24, φ=np.pi / 2, tb_model=tb)
add_hoppings_by_graph_distance(tb, 0, stagger_mass, is_hermitian=False)
add_hoppings_by_graph_distance(tb, 1, -1.0)
add_hoppings_by_graph_distance(tb, 2, haldane_amp, is_hermitian=True)
```

`haldane_nnn_hopping_amplitude` finds the unique common nearest neighbour of
`i` and `j`, computes the chirality `ν = sign[(r_k - r_i) × (r_j - r_k)] = ±1`,
and returns `t2 * exp(i φ ν)`.  (Note the two constructions are related by a
global chirality flip; the explicit-template form above is the one used for the
reference Chern numbers in the tests.)

## 5. Uniform grids (`uniform_grids.py`)

```python
@dataclass
class Uniform_Grids:
    name: str
    dim: int
    sample_size: list[int]
    basis_vec_list: list[list[float]]
    cell_volume: float
    twisted_phases_over_2π: list[float]
    site_int_list: list[tuple[int, ...]]
    site_int_to_index_map: dict[tuple[int, ...], int]     # 1-based
    site_crys_list: list[np.ndarray]
    site_cart_list: list[np.ndarray]
    nsite: int
```

Grid points are enumerated first-axis-fastest and their crystal coordinates are

```text
site_crys[d] = (site_int[d] + twisted_phases_over_2π[d]) / sample_size[d]
```

i.e. the twist shifts the grid off the high-symmetry points — inserting a flux
`θ_d` is equivalent to shifting the crystal-momentum grid by `θ_d / L_d`.

Two constructors exist (see §9e for why they are split):

```python
initialize_uniform_grids(*, basis_vec_list, sample_size, name="", twisted_phases_over_2π)
initialize_uniform_grids_from_lattice(r_data, *, twisted_phases_over_2π=None)
```

`initialize_uniform_grids_from_lattice` uses the reciprocal vectors
`2π inv(brav_rows).T` as basis vectors and defaults the twist to the lattice's
stored `twisted_phases_over_2π`.

## 6. Utilities (`utils.py`)

- `dual_basis_vec_mat` / `dual_basis_vec_list` — the 2π-inverse-transpose dual
  basis (§2).
- `build_Hk_crys(tb_model)` — returns `Hk_crys(k_crys)` in the **periodic
  gauge** `H(k + G) = H(k)` (the Bloch phase uses only the integer cell shift
  `Δ_cell`, not the sublattice offsets).  When `input_hopping_map` is non-empty
  the infinite-system templates are used directly; otherwise the
  `full_hopping_map` torus hoppings are compressed into one template per
  `(sub_from, sub_to, Δ)` and divided by `n_cell`.
- `Chern_number_Fukui_Hatsugai_Suzuki(Hk_crys, *, band, nk=51)` — single-band
  Fukui–Hatsugai–Suzuki Chern number (1-based `band`).
- `generate_bilinear_terms(tb_model, *, twisted_phases_over_2π=None)` — returns
  `(i_site, j_site, amplitude)` triples (1-based) with twisted-boundary phases
  `exp(i 2π Σ_d θ_d w_d)` attached to bonds crossing a periodic boundary with
  winding `w_d`.
- `build_real_space_tb_Hamiltonain(tb_model, *, twisted_phases_over_2π=None)` —
  the `n_site × n_site` real-space Hamiltonian as a `scipy.sparse.csc_matrix`.
- `many_body_Chern_number_Fukui_Hatsugai_Suzuki(tb_model, *, n_occ, nθ=21)` —
  many-body Chern number on the flux torus `(θ₁, θ₂) ∈ [0,1]²` (requires PBC in
  all directions).  For a non-interacting model it equals the sum of the
  single-particle Chern numbers of the occupied bands, so `n_occ = n_cell`
  returns one filled band's Chern number (half filling for a two-band model).
  This routine uses a two-dimensional flux grid and dense diagonalization;
  it is intended for non-interacting systems with a gapped occupied subspace.

## 7. Band plots (`band_plot.py`)

```python
plot_bands(Hk_crys, k_data, *, k_path, k_path_name_list=None,
           nband_range=range(1, 2), nk=30, save_path=None) -> (fig, ax)
```

plots the band structure along a crystal-coordinate `k_path`, scaling the
x-axis by the Cartesian length of each segment and drawing vertical ticks at the
turning points.  `nband_range` is 1-based.

```python
plot_band_contour(hk_cart, k_data, *, k_cart_ranges=None, levels=10, band_idx=1,
                  normalize_k_cart_range_with_lattice_constant=True,
                  show_BZ=True, show_band_width_band_gap_info=True,
                  save_path=None)
```

plots a 2D band contour (1-based `band_idx`) with an optional first-Brillouin-zone
outline.  `find_1st_BZ_k_cart_list(reciprocal_vec_list, *, max_shell=3)` returns
the BZ vertices (Wigner–Seitz cell) by intersecting the perpendicular bisectors
of the shortest reciprocal vectors.

The two lattice/model plotters

```python
plot_real_space_lattice(lattice, *, save_path=None) -> (fig, ax)
plot_real_space_tightbinding_model(tb_model, *, save_path=None) -> (fig, ax)
```

draw sites (coloured by sublattice), bulk vs. wrapped (ghost) bonds, the unit
cell and arrowed Bravais vectors; the model plot overlays hopping arcs with
amplitude labels.

## 8. Design invariants (cross-checks)

| Quantity | Honeycomb `[3, 4]` PBC |
|---|---|
| `n_site` | 24 |
| `n_cell` | 12 |
| `n_sub` | 2 |
| `cell_volume` | `√3 / 2 = 0.8660254037844386` |
| `sub_name_list` | `["A1", "A2"]` |
| `site_list[:6]` | `((0,0),1),((0,0),2),((1,0),1),((1,0),2),((2,0),1),((2,0),2)` |
| graph `n_edges` | 36 |
| NN distance | `1/√3` |

Haldane model (`t1=-1`, `t2=-0.24`, `φ=π/2`, `M=0.7`, the §4.3 templates):
eigenvalues at `Γ=(0,0)`, `(1/3,1/3)`, `K=(2/3,1/3)`, `M=(1/2,1/2)` are
`±3.080584360150`, `±1.868154169227`, `±0.547076581450`, `±1.220655561573`,
and the Chern numbers are `C(band1) = -1`, `C(band2) = +1`.

## 9. Differences from the Julia source (with reasons)

The port is *faithful* in data structures, algorithms, and indexing (1-based
sites, first-axis-fastest ordering, periodic gauge, PBC minimum-image wrapping),
but the language forces a handful of deliberate differences.

### (a) Graphs: `igraph.Graph` behind `LatticeGraph` vs `Graphs.SimpleGraph`

Julia stores the nearest-neighbour graph as a `Graphs.SimpleGraph`; Python wraps
an **`igraph.Graph`** inside a thin `LatticeGraph` adapter. Exposing the
underlying `igraph_graph` object gives access to igraph operations such as
community detection, isomorphism, layouts, and GML/GraphML export.  The
adapter re-implements the small `Graphs` API subset the package uses
(`neighbors`, `gdistances`, `edges`) in **1-based** indices so that all
downstream code stays identical to the Julia logic.

### (b) Plotting: matplotlib vs CairoMakie

Julia plots with `CairoMakie`; Python uses **matplotlib**.  The plotting
functions reproduce the same *content* (sites coloured by sublattice, solid
bulk vs. dashed wrapped bonds, ghost sites at unwrapped positions, unit cell and
Bravais arrows, band ticks at the path turning points) but not pixel-identical
styling. The Python plotters **return `(fig, ax)`** for customization
(the Julia versions return a `Figure`); scripts can display them with
`matplotlib.pyplot.show()`.

### (c) Sparse matrices: scipy vs Julia `SparseArrays`

`build_real_space_tb_Hamiltonain` returns a **`scipy.sparse.csc_matrix`**, the
direct analogue of Julia's `SparseMatrixCSC`.  Everything downstream
(`.toarray()`, `.nnz`) is the standard scipy API.

### (d) The mutation suffix `!` is dropped

Julia convention marks mutating functions with `!` (`add_hopping_term!`,
`_wrap_Δ_crys!`, `_compute_winding!`).  Python has no such convention, so the
suffix is dropped (`add_hopping_term`, ...).  In-place behaviour is instead
documented per function in its docstring ("mutated in place").
`_wrap_Δ_crys` returns a new array. `_compute_winding` mutates its input
destination-cell list and returns the winding numbers (or `None` for a
hopping that leaves an open boundary).

### (e) `initialize_uniform_grids_from_lattice` is a separate function

Julia defines two *methods* of `initialize_uniform_grids` (one taking
`basis_vec_list` + `sample_size`, one taking `r_data::Real_Space_Lattice`)
selected by multiple dispatch. This Python API instead uses
**distinct names**: `initialize_uniform_grids` and
`initialize_uniform_grids_from_lattice`.

### (f) Exceptions (`ValueError`) instead of `@assert` / `error`

Julia uses `@assert` and `error` for input validation; Python raises
**`ValueError`** (or keeps `assert` only for internal numerical invariants such
as the dual-basis identity and the many-body PBC precondition).  Examples: a
twist on an open direction, a non-2D lattice passed to a 2D plotter, or an
out-of-range sublattice index all raise `ValueError`. The Chern routines
explicitly symmetrize their
Hamiltonians before diagonalizing; they do not reject non-Hermitian inputs.

### (g) Numeric types: `float` / `np.float64` vs generic `T`

Julia is generic over `T` (the lattice may hold symbolic `MathExpr` entries).
Python stores **`float` / `np.float64`** throughout (`brav_vec_list`,
`cell_volume`, `site_crys_list` as `np.ndarray`).  Symbolic lattices are not
supported: conversion of Bravais vectors and sublattice positions to floats
can fail before graph construction. The `graph = None` fallback only covers
`TypeError` or `ValueError` raised inside graph construction.

### (h) Parameter-key naming lives in user code, not the package

The package **never forces key names** — its public keyword arguments include both
ASCII and Unicode identifiers (`pbc_indicator`, `twisted_phases_over_2π`,
`φ`, `nθ`, ...).  The
translation of mathematical symbols to ASCII keys (`t′` → `"t'"`,
`ϕ_over_2π` → `"phi_over_2pi"`, `t′′` → `"t_double_prime"`) is a convention used
only in the *examples* and *tests* (their parameter dictionaries), so users are
free to name keys however they like.

### (i) `plot_band_contour` is provided; `plot_band_counter` is not

The Julia module **exports** `plot_band_counter` — a typo — while *defining*
`plot_band_contour`.  The Python port deliberately provides only the correctly
spelled `plot_band_contour` (and no `plot_band_counter`).

### (j) The Haldane helper keeps its name

`haldane_nnn_hopping_amplitude(i_site, j_site; t2=0.3, φ=π/2, tb_model)` is
named **identically** to the Julia original (including the keyword `φ`), so the
helper is discoverable across the two codebases.

### (k) Two Haldane chirality conventions coexist (in both languages)

The Julia package itself defines the Haldane NNN hopping in two equivalent-but-
opposite ways, and the port reproduces both faithfully:

- `haldane_nnn_hopping_amplitude` computes the chirality
  $\nu = \mathrm{sign}[(\mathbf{r}_k - \mathbf{r}_i) \times (\mathbf{r}_j - \mathbf{r}_k)]$
  from the *geometry* of the two NN hops through the common neighbour;
- the explicit template convention used by the FCI model builders
  (e.g. `RealSpace_ExactDiagonalization`'s `bosonic_fci.jl`, `TightBinding`'s
  own `test.jl`) assigns the phases `±iφ` by hand.

The two conventions reverse the hopping chirality and hence the Chern signs,
i.e. $C \to -C$: the same
Haldane model built via `add_hoppings_by_graph_distance(tb, 2,
haldane_nnn_hopping_amplitude(...))` has Chern numbers $(+1,-1)$, while the
template version has $(-1,+1)$.  This is **not** a porting discrepancy — the
same sign flip occurs when comparing the two Julia constructions — but users
comparing Chern numbers across constructions should pin the convention
explicitly.

## 10. Runnable consistency check

Run this cell with `tightbinding_py` installed. It checks the honeycomb
geometry and reciprocal-basis convention described above.


In [ ]:
import numpy as np
from tightbinding_py import initialize_real_space_lattice, dual_basis_vec_mat

lat = initialize_real_space_lattice(
    lattice_name="honeycomb", sample_size=[3, 4], pbc_indicator=[True, True],
)
assert (lat.n_cell, lat.n_sub, lat.n_site) == (12, 2, 24)
assert len(lat.graph.edges()) == 36
assert lat.site_list[:4] == [((0, 0), 1), ((0, 0), 2), ((1, 0), 1), ((1, 0), 2)]
np.testing.assert_allclose(lat.cell_volume, np.sqrt(3) / 2)
basis_columns = np.asarray(lat.brav_vec_list).T
reciprocal_columns = dual_basis_vec_mat(basis_columns)
np.testing.assert_allclose(
    reciprocal_columns.T @ basis_columns, 2 * np.pi * np.eye(2), atol=1e-12,
)
print("Honeycomb geometry, indexing, and reciprocal basis verified.")
